
### This notebook is meant to train novice and expert models on Google Colab 

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(' Google Drive mounted')

## Install Dependencies

In [ ]:
!pip install torch torchvision tqdm numpy scikit-learn -q
print('Dependencies installed')

##  Copy Project Files from Drive

In [ ]:
import shutil
import os

project_path = '/content/drive/MyDrive/FinalProj-DL'
work_path = '/content/work/FinalProj-DL'

experiment_name = 'v2_temporal_signed'
checkpoints_dir = f'checkpoints/{experiment_name}'
eval_file = f'evaluation_results_{experiment_name}.txt'
drive_output = f'/content/drive/MyDrive/FinalProj-DL-Results/{experiment_name}'

#copy project files
if os.path.exists(project_path):
    shutil.copytree(project_path, work_path, dirs_exist_ok=True)
    os.chdir(work_path)
    print(f'Project copied to {work_path}')
    print(f'Current directory: {os.getcwd()}')
    print(f'Experiment: {experiment_name}')
    print('\nDirectory structure:')
    os.system('ls -la')
else:
    print('ERROR: FinalProj-DL not found in Google Drive')
    print('Please upload FinalProj-DL folder to: /My Drive/FinalProj-DL/')

##  Train Novice Model (15 epochs)

In [ ]:
import subprocess

print('Starting Novice Model Training')
result = subprocess.run([
    'python', 'src/train.py',
    '--dataset_path', 'data/raw/novice_dataset.npz',
    '--save_dir', checkpoints_dir,
    '--save_name', 'novice',
    '--epochs', '15',
    '--batch_size', '32',
    '--sequence_length', '8'
])
print('Novice training complete')

##  Train Expert Model (20 epochs with hyperparameter tuning)

In [ ]:
print('\nStarting Expert Model Training:')
result = subprocess.run([
    'python', 'src/train.py',
    '--dataset_path', 'data/raw/expert_dataset.npz',
    '--save_dir', checkpoints_dir,
    '--save_name', 'expert',
    '--epochs', '20',
    '--batch_size', '32',
    '--sequence_length', '8',
    '--learning_rate', '0.0005',
    '--dropout', '0.1'
])
print('Expert training complete')

##  Test Trained Models

In [ ]:
print('Testing trained models...')
result = subprocess.run([
    'python', 'src/test_trained_model.py',
    '--novice_checkpoint', f'{checkpoints_dir}/novice_best.pth',
    '--expert_checkpoint', f'{checkpoints_dir}/expert_best.pth',
    '--sequence_length', '8',
    '--output_file', eval_file,
])
print('Testing complete')

##  Download Trained Models to Google Drive

In [ ]:
#create output folder in Drive
os.makedirs(drive_output, exist_ok=True)

#copy checkpoints for this experiment
if os.path.exists(checkpoints_dir):
    for file in os.listdir(checkpoints_dir):
        src = os.path.join(checkpoints_dir, file)
        dst = os.path.join(drive_output, file)
        shutil.copy(src, dst)
        print(f'Downloaded {file}')
else:
    print(f'No checkpoints found at {checkpoints_dir}!')

#copy evaluation results
if os.path.exists(eval_file):
    shutil.copy(eval_file, os.path.join(drive_output, os.path.basename(eval_file)))
    print(f'Downloaded {eval_file}')
else:
    print(f'No evaluation file found at {eval_file}!')

print(f'\nAll files saved to: {drive_output}')

##  View Checkpoint Files

In [ ]:
!ls -lh "{checkpoints_dir}"
!ls -lh "{eval_file}"
!ls -lh "{drive_output}"